# SaulLM Quantization Demo (Colab)

This notebook runs end-to-end latency + accuracy benchmarking for FP16, 8-bit, and 4-bit modes, and prints each model response for the provided NDA file.

In [ ]:
# Step 1: install dependencies (Colab)
!pip install -q -r requirements.txt

In [ ]:
# Step 2: (optional) verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Step 3: run benchmark script (Colab-safe default first: 4-bit + 8-bit)
!python scripts/run_benchmark.py --precisions 4-bit,8-bit --max-new-tokens 96 --max-gpu-memory 12GiB

# Optional: try baseline FP16 with heavier CPU offload (can be much slower)
# !python scripts/run_benchmark.py --precisions baseline --max-new-tokens 96 --max-gpu-memory 10GiB --max-cpu-memory 64GiB

In [ ]:
# Step 4: display telemetry and accuracy tables
import pandas as pd
lat_df = pd.read_csv('outputs/metrics_log.csv')
acc_df = pd.read_csv('outputs/accuracy_log.csv')
display(lat_df)
display(acc_df)

In [ ]:
# Step 5: visualize latency per phase and precision
import matplotlib.pyplot as plt
pivot = lat_df.pivot(index='Phase', columns='Precision', values='Time_sec')
pivot.plot(kind='bar', figsize=(10, 5), title='Latency by Stage and Precision')
plt.ylabel('Seconds')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Step 6: print each NDA response for demo narration
import json
from pathlib import Path

responses = json.loads(Path('outputs/demo_responses.json').read_text())

for item in responses:
    print('=' * 100)
    print('Precision:', item['precision'])
    print('Accuracy:', item['accuracy'])
    # Fixed the newline error below by using \n
    print('Response:\n', item['response'])


In [ ]:
# Step 7: optional architecture profile (parameters + memory estimate + layer stats)
from src.engine.model_loader import load_model_and_tokenizer
from src.telemetry.model_profiler import profile_model

model, tokenizer = load_model_and_tokenizer(precision='4bit')
stats = profile_model(model, print_full_layers=False)
stats